# Wisconsin Diagnostic Cancer Dataset - Análise Exploratória

### Resumo
A análise exploratória do Wisconsin Diagnostic Cancer Dataset permitiu identificar diversas correlações entre as características dos núcleos celulares e o diagnóstico (Maligno ou Benigno). Variáveis como 'concave points_worst', 'perimeter_worst' e 'radius_worst' apresentaram correlações positivas fortes com o diagnóstico maligno, indicando que valores mais elevados nessas características estão associados a uma maior probabilidade de malignidade. As visualizações confirmam que as distribuições dessas características diferem significativamente entre os casos benignos e malignos, validando sua importância para fins de diagnóstico e classificação.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import os
import seaborn as sns

os.makedirs("graficos", exist_ok=True)
os.makedirs("tabelas", exist_ok=True)

In [ ]:
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv(os.path.join(path, "data.csv"))
df.head().to_csv("tabelas/wisconsin_primeiras_linhas.csv", index=False)
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().to_csv("tabelas/wisconsin_estatisticas.csv")
df.describe()

In [ ]:
null_counts = df.isnull().sum()
null_counts.to_csv("tabelas/wisconsin_valores_nulos.csv", header=["valores_nulos"])
print(null_counts)

In [ ]:
df = df.drop(columns=["Unnamed: 32"], errors="ignore")

In [ ]:
diag_counts = df["diagnosis"].value_counts()
diag_counts.to_csv("tabelas/wisconsin_distribuicao_diagnostico.csv", header=["contagem"])
print(diag_counts)

In [ ]:
plt.figure(figsize=(6, 4))
df["diagnosis"].value_counts().plot(kind="bar", color=["steelblue", "salmon"])
plt.title("Distribuição do Diagnóstico")
plt.xlabel("Diagnóstico")
plt.ylabel("Contagem")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("graficos/wisconsin_distribuicao_diagnostico.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
features = df.columns[2:32]
feature_groups = {
    "mean": [f for f in features if f.endswith("_mean")],
    "se": [f for f in features if f.endswith("_se")],
    "worst": [f for f in features if f.endswith("_worst")]
}

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(feature_groups["mean"]):
    row, col = i // 5, i % 5
    ax = axes[row, col]
    ax.hist(df[feat], bins=30, color="steelblue", edgecolor="black", alpha=0.7)
    ax.set_title(feat.replace("_mean", ""))
plt.suptitle("Distribuição das Variáveis Mean", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_histogramas_mean.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(feature_groups["se"]):
    row, col = i // 5, i % 5
    ax = axes[row, col]
    ax.hist(df[feat], bins=30, color="salmon", edgecolor="black", alpha=0.7)
    ax.set_title(feat.replace("_se", ""))
plt.suptitle("Distribuição das Variáveis SE", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_histogramas_se.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, feat in enumerate(feature_groups["worst"]):
    row, col = i // 5, i % 5
    ax = axes[row, col]
    ax.hist(df[feat], bins=30, color="darkorange", edgecolor="black", alpha=0.7)
    ax.set_title(feat.replace("_worst", ""))
plt.suptitle("Distribuição das Variáveis Worst", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_histogramas_worst.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
corr_matrix = df[features].corr()
plt.figure(figsize=(18, 15))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, fmt=".1f", linewidths=0.5)
plt.title("Matriz de Correlação")
plt.tight_layout()
plt.savefig("graficos/wisconsin_matriz_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
df_encoded = df.copy()
df_encoded["diagnosis"] = df_encoded["diagnosis"].map({"M": 1, "B": 0})
corr_with_diagnosis = df_encoded[features.tolist() + ["diagnosis"]].corr()["diagnosis"].drop("diagnosis")
corr_with_diagnosis.sort_values(ascending=False).to_csv("tabelas/wisconsin_correlacao_diagnostico.csv", header=["correlacao"])
print(corr_with_diagnosis.sort_values(ascending=False))

In [ ]:
top_features = corr_with_diagnosis.abs().sort_values(ascending=False).head(10).index
plt.figure(figsize=(10, 6))
corr_with_diagnosis[top_features].plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Top 10 Correlações com Diagnóstico")
plt.ylabel("Correlação")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("graficos/wisconsin_top10_correlacoes.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(["radius_mean", "texture_mean", "concavity_mean"]):
    ax = axes[i]
    for diag in ["M", "B"]:
        subset = df[df["diagnosis"] == diag]
        ax.hist(subset[feat], bins=30, alpha=0.6, label=diag)
    ax.set_title(feat.replace("_mean", ""))
    ax.legend()
plt.suptitle("Distribuição por Diagnóstico - Top Features", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_distribuicao_top_features.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pairs = [("radius_mean", "texture_mean"), ("perimeter_mean", "area_mean"), ("concavity_mean", "compactness_mean")]
for i, (x, y) in enumerate(pairs):
    ax = axes[i]
    for diag, color in [("M", "salmon"), ("B", "steelblue")]:
        subset = df[df["diagnosis"] == diag]
        ax.scatter(subset[x], subset[y], c=color, alpha=0.5, label=diag)
    ax.set_xlabel(x.replace("_mean", ""))
    ax.set_ylabel(y.replace("_mean", ""))
    ax.legend()
plt.suptitle("Relações entre Features por Diagnóstico", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_scatter_features.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
mean_feats = feature_groups["mean"][:6]
for i, feat in enumerate(mean_feats):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    df.boxplot(column=feat, by="diagnosis", ax=ax)
    ax.set_title(feat.replace("_mean", ""))
    ax.set_xlabel("Diagnóstico")
plt.suptitle("Boxplots por Diagnóstico - Variáveis Mean", fontsize=14)
plt.tight_layout()
plt.savefig("graficos/wisconsin_boxplots_diagnostico.png", dpi=150, bbox_inches="tight")
plt.show()